# Hipotesis

Este notebook valida las hipótesis planteadas al final de `reports/01_eda/ANALYSIS_20260301.md`.

Hipótesis a validar:
1. Nulos estructurales por fuente de captura.
2. Señal no lineal entre duración y popularidad.
3. Coexistencia de poblaciones (musical vs no musical) en extremos de duración+título.
4. Valor real de features temporales + agregados con control anti-leakage.


## Importacion de datos y librerias


In [ ]:
# Librerías base
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import chi2_contingency, kruskal
from sklearn.metrics import roc_auc_score, average_precision_score, mutual_info_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# Configuración visual
plt.style.use("default")
sns.set_palette("deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)


In [ ]:
# Utilidades de carga y validación

def get_latest_file(dir_path, pattern):
    files = sorted(Path(dir_path).glob(pattern), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos con patrón {pattern} en {dir_path}")
    return files[-1]


def safe_rate(series):
    s = pd.Series(series).dropna()
    if len(s) == 0:
        return np.nan
    return float((s.astype(int) == 1).mean())


def build_presence_table(df, cols, group_col="source"):
    rows = []
    for source, grp in df.groupby(group_col, dropna=False):
        row = {group_col: source, "n_rows": len(grp)}
        for c in cols:
            present = grp[c].notna().mean()
            row[f"{c}_present_pct"] = present * 100
            row[f"{c}_missing_pct"] = (1 - present) * 100
        rows.append(row)
    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)


def cramers_v_from_crosstab(ct):
    chi2, pvalue, _, _ = chi2_contingency(ct)
    n = ct.to_numpy().sum()
    r, k = ct.shape
    if n == 0 or min(r - 1, k - 1) <= 0:
        return np.nan, pvalue, chi2
    return np.sqrt((chi2 / n) / min(r - 1, k - 1)), pvalue, chi2


def bootstrap_rate_ci(y, n_boot=300, alpha=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y).astype(int)
    if len(y) == 0:
        return np.nan, np.nan, np.nan
    vals = []
    for _ in range(n_boot):
        sample = rng.choice(y, size=len(y), replace=True)
        vals.append(sample.mean())
    vals = np.sort(vals)
    lo = np.quantile(vals, alpha / 2)
    hi = np.quantile(vals, 1 - alpha / 2)
    return float(y.mean()), float(lo), float(hi)


In [ ]:
# Funciones para hipótesis 2, 3 y 4

def compute_duration_nonlinear_metrics(df, target_col="popular_high"):
    tmp = df[["duration_min", "track_popularity", target_col]].dropna().copy()
    if tmp.empty:
        raise ValueError("No hay datos suficientes para métricas no lineales de duración")

    pearson_pop = tmp["duration_min"].corr(tmp["track_popularity"], method="pearson")
    spearman_pop = tmp["duration_min"].corr(tmp["track_popularity"], method="spearman")
    pearson_target = tmp["duration_min"].corr(tmp[target_col], method="pearson")
    spearman_target = tmp["duration_min"].corr(tmp[target_col], method="spearman")

    # Discretización por cuantiles (fallback robusto)
    try:
        qbin = pd.qcut(tmp["duration_min"], q=10, duplicates="drop")
    except ValueError:
        qbin = pd.cut(tmp["duration_min"], bins=10)

    qbin_codes = qbin.astype(str)
    mi = mutual_info_score(tmp[target_col].astype(int), qbin_codes)

    # Bins interpretables
    bins = [-np.inf, 2, 3, 4, 6, np.inf]
    labels = ["<=2", "2-3", "3-4", "4-6", ">6"]
    tmp["duration_bin"] = pd.cut(tmp["duration_min"], bins=bins, labels=labels)

    groups = [g["track_popularity"].values for _, g in tmp.groupby("duration_bin", observed=False) if len(g) > 0]
    if len(groups) >= 2:
        kw_stat, kw_p = kruskal(*groups)
    else:
        kw_stat, kw_p = np.nan, np.nan

    return {
        "pearson_popularity": pearson_pop,
        "spearman_popularity": spearman_pop,
        "pearson_target": pearson_target,
        "spearman_target": spearman_target,
        "mutual_information": mi,
        "kruskal_stat": kw_stat,
        "kruskal_pvalue": kw_p,
        "df": tmp,
    }


def tag_duration_extremes(df):
    out = df.copy()
    out["extreme_short"] = out["duration_min"] <= 0.1
    out["extreme_long"] = out["duration_min"] >= 15
    out["extreme_flag"] = out["extreme_short"] | out["extreme_long"]
    return out


def build_train_only_entity_aggregates(train_df, full_df):
    artist_stats = (
        train_df.groupby("artist_id", dropna=False)
        .agg(
            artist_avg_popularity_train=("track_popularity", "mean"),
            artist_popularity_std_train=("track_popularity", "std"),
            artist_track_count_train=("track_id", "count"),
            artist_avg_duration_train=("duration_min", "mean"),
        )
        .reset_index()
    )

    playlist_stats = (
        train_df.groupby("playlist_id", dropna=False)
        .agg(
            playlist_avg_popularity_train=("track_popularity", "mean"),
            playlist_popularity_std_train=("track_popularity", "std"),
            playlist_track_count_train=("track_id", "count"),
            playlist_avg_duration_train=("duration_min", "mean"),
        )
        .reset_index()
    )

    merged = full_df.merge(artist_stats, on="artist_id", how="left")
    merged = merged.merge(playlist_stats, on="playlist_id", how="left")
    return merged


def evaluate_model_variants_cv(df, variants, n_splits=5, random_state=42):
    splitter = StratifiedShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=random_state)

    model_defs = {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=random_state),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            random_state=random_state,
            n_jobs=-1,
        ),
    }

    records = []

    for split_id, (tr_idx, te_idx) in enumerate(splitter.split(df, df["popular_high"])):
        train = df.iloc[tr_idx].copy()
        test = df.iloc[te_idx].copy()

        # Para variante V3, construir agregados solo con train y aplicar a train/test
        # sin depender del índice original (los merge pueden resetearlo)
        train = train.copy()
        test = test.copy()
        train["__row_id"] = np.arange(len(train))
        test["__row_id"] = np.arange(len(test))
        train["__set"] = "train"
        test["__set"] = "test"

        train_test_concat = pd.concat([train, test], axis=0, ignore_index=True)
        with_train_only = build_train_only_entity_aggregates(train, train_test_concat)
        train_v3 = with_train_only[with_train_only["__set"] == "train"].copy()
        test_v3 = with_train_only[with_train_only["__set"] == "test"].copy()

        for variant_name, feat_cols in variants.items():
            if variant_name == "V3_con_agregados_train_only":
                tr_data = train_v3.copy()
                te_data = test_v3.copy()
            else:
                tr_data = train.copy()
                te_data = test.copy()

            X_train = tr_data[feat_cols].copy()
            X_test = te_data[feat_cols].copy()
            y_train = tr_data["popular_high"].astype(int)
            y_test = te_data["popular_high"].astype(int)

            # Imputación por mediana del train
            for c in feat_cols:
                med = X_train[c].median()
                X_train[c] = X_train[c].fillna(med)
                X_test[c] = X_test[c].fillna(med)

            for model_name, model in model_defs.items():
                model.fit(X_train, y_train)
                proba = model.predict_proba(X_test)[:, 1]

                records.append(
                    {
                        "split_id": split_id,
                        "variant": variant_name,
                        "model": model_name,
                        "roc_auc": roc_auc_score(y_test, proba),
                        "pr_auc": average_precision_score(y_test, proba),
                    }
                )

    return pd.DataFrame(records)


In [ ]:
# Rutas y carga de datasets
BASE_DIR = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"

raw_file = get_latest_file(RAW_DIR, "spotify_tracks_merged_raw_*.parquet")
interim_file = get_latest_file(INTERIM_DIR, "spotify_tracks_interim_*.parquet")

print(f"Usando RAW: {raw_file}")
print(f"Usando INTERIM: {interim_file}")

raw_df = pd.read_parquet(raw_file)
interim_df = pd.read_parquet(interim_file)

print("\nShapes")
print("raw_df:", raw_df.shape)
print("interim_df:", interim_df.shape)


In [ ]:
# Checks de contrato
raw_required = {"source", "search_query", "search_market", "playlist_id", "playlist_name", "added_at"}
interim_required = {
    "track_id", "track_popularity", "duration_ms", "popular_high",
    "artist_avg_popularity", "playlist_avg_popularity", "playlist_popularity_std",
    "artist_popularity_std", "years_since_release", "album_release_year",
    "playlist_track_count", "artist_track_count", "artist_avg_duration", "playlist_avg_duration",
}

missing_raw = raw_required - set(raw_df.columns)
missing_interim = interim_required - set(interim_df.columns)

if missing_raw:
    raise ValueError(f"Faltan columnas en raw_df: {sorted(missing_raw)}")
if missing_interim:
    raise ValueError(f"Faltan columnas en interim_df: {sorted(missing_interim)}")

if set(interim_df["popular_high"].dropna().unique()) - {0, 1}:
    raise ValueError("popular_high debe ser binaria (0/1)")

interim_df["duration_min"] = interim_df["duration_ms"] / 60000

print("Checks de contrato: OK")


## Hipótesis 1 - Nulos estructurales por fuente de captura

**Hipótesis**: Los nulos en `search_*` y `playlist_*` responden a distintas fuentes de captura y `added_at` aplica solo a un subconjunto.


In [ ]:
h1_cols = ["search_query", "search_market", "playlist_id", "playlist_name", "added_at"]

presence_tbl = build_presence_table(raw_df, h1_cols, group_col="source")
print("Tabla de presencia/ausencia por source:")
display(presence_tbl)


In [ ]:
# Contingencias + Chi-cuadrado + Cramér's V
h1_stats = []
for col in h1_cols:
    ct = pd.crosstab(raw_df["source"], raw_df[col].notna())
    cv, pval, chi2 = cramers_v_from_crosstab(ct)
    h1_stats.append({
        "columna": col,
        "chi2": chi2,
        "pvalue": pval,
        "cramers_v": cv,
    })

h1_stats_df = pd.DataFrame(h1_stats).sort_values("cramers_v", ascending=False)
print("Asociación entre source y presencia de columnas:")
display(h1_stats_df)


In [ ]:
# Heatmap de % not null por source/columna
heat_data = (
    raw_df.groupby("source", dropna=False)[h1_cols]
    .apply(lambda x: x.notna().mean() * 100)
    .sort_index()
)

plt.figure(figsize=(10, 4))
sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "% not null"})
plt.title("H1 - % de presencia por source")
plt.xlabel("Columnas")
plt.ylabel("Source")
plt.tight_layout()
plt.show()


In [ ]:
# Barras apiladas de presencia/ausencia por columna
fig, axes = plt.subplots(1, len(h1_cols), figsize=(20, 4), sharey=True)

for i, col in enumerate(h1_cols):
    tmp = pd.crosstab(raw_df["source"], raw_df[col].notna(), normalize="index") * 100
    tmp = tmp.rename(columns={False: "Missing", True: "Present"})
    tmp[["Missing", "Present"]].plot(kind="bar", stacked=True, ax=axes[i], color=["#d95f02", "#1b9e77"])
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("%" if i == 0 else "")
    axes[i].legend(loc="lower right", fontsize=8)

plt.suptitle("H1 - Presencia/Ausencia por source", y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
# Decisión H1
h1_valid = bool((h1_stats_df["pvalue"] < 0.05).all() and (h1_stats_df["cramers_v"].fillna(0) > 0.2).all())

print("Criterio:")
print("- p-value < 0.05 en todas las columnas")
print("- Cramér's V > 0.2 en todas las columnas")
print(f"\nResultado H1: {'VALIDADA' if h1_valid else 'PARCIAL / NO EVIDENCIA'}")


## Hipótesis 2 - Relaciones no lineales duración-popularidad

**Hipótesis**: Existen relaciones no lineales o segmentadas entre duración y popularidad que Pearson no captura.


In [ ]:
h2 = compute_duration_nonlinear_metrics(interim_df, target_col="popular_high")

metrics_h2 = pd.DataFrame(
    {
        "metrica": [
            "pearson(duration, track_popularity)",
            "spearman(duration, track_popularity)",
            "pearson(duration, popular_high)",
            "spearman(duration, popular_high)",
            "mutual_information(popular_high, duration_qbins)",
            "kruskal_stat(track_popularity ~ duration_bins)",
            "kruskal_pvalue",
        ],
        "valor": [
            h2["pearson_popularity"],
            h2["spearman_popularity"],
            h2["pearson_target"],
            h2["spearman_target"],
            h2["mutual_information"],
            h2["kruskal_stat"],
            h2["kruskal_pvalue"],
        ],
    }
)

display(metrics_h2)


In [ ]:
# Tasa de popular_high por bins interpretables con bootstrap CI
h2_df = h2["df"].copy()

def rate_ci_by_bin(df, bin_col="duration_bin", y_col="popular_high"):
    rows = []
    for b, grp in df.groupby(bin_col, observed=False):
        rate, lo, hi = bootstrap_rate_ci(grp[y_col], n_boot=300, alpha=0.05, random_state=42)
        rows.append({"duration_bin": str(b), "n": len(grp), "rate": rate, "ci_low": lo, "ci_high": hi})
    out = pd.DataFrame(rows)
    order = ["<=2", "2-3", "3-4", "4-6", ">6"]
    out["duration_bin"] = pd.Categorical(out["duration_bin"], categories=order, ordered=True)
    return out.sort_values("duration_bin")

h2_rate_tbl = rate_ci_by_bin(h2_df)
display(h2_rate_tbl)


In [ ]:
# Visualizaciones H2
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# (1) Rate by bin + CI
plot_tbl = h2_rate_tbl.dropna(subset=["rate"]).copy()
axes[0].plot(plot_tbl["duration_bin"].astype(str), plot_tbl["rate"], marker="o", color="#1f77b4")
axes[0].fill_between(
    plot_tbl["duration_bin"].astype(str),
    plot_tbl["ci_low"],
    plot_tbl["ci_high"],
    alpha=0.2,
    color="#1f77b4",
)
axes[0].set_title("Tasa popular_high por bin de duración")
axes[0].set_xlabel("Duración (min)")
axes[0].set_ylabel("Tasa")
axes[0].grid(alpha=0.3)

# (2) Boxplot popularidad por bin
sns.boxplot(data=h2_df, x="duration_bin", y="track_popularity", ax=axes[1], color="#8da0cb")
axes[1].set_title("track_popularity por bin de duración")
axes[1].set_xlabel("Duración (min)")
axes[1].set_ylabel("track_popularity")
axes[1].grid(alpha=0.3)

# (3) Scatter + rolling mean
sample_scatter = h2_df.sample(min(len(h2_df), 15000), random_state=42)
axes[2].scatter(sample_scatter["duration_min"], sample_scatter["track_popularity"], s=8, alpha=0.15, color="#666666")
roll = h2_df[["duration_min", "track_popularity"]].sort_values("duration_min")
roll["rolling_mean"] = roll["track_popularity"].rolling(window=500, min_periods=100).mean()
axes[2].plot(roll["duration_min"], roll["rolling_mean"], color="red", linewidth=2)
axes[2].set_title("Scatter + rolling mean")
axes[2].set_xlabel("Duración (min)")
axes[2].set_ylabel("track_popularity")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Decisión H2
spearman_gain = abs(h2["spearman_popularity"]) - abs(h2["pearson_popularity"])
h2_valid = bool(
    (h2["mutual_information"] > 0)
    and (
        (spearman_gain > 0.01)
        or (pd.notna(h2["kruskal_pvalue"]) and h2["kruskal_pvalue"] < 0.05)
    )
)

print(f"MI: {h2['mutual_information']:.6f}")
print(f"|Spearman|-|Pearson| (popularity): {spearman_gain:.6f}")
print(f"Kruskal p-value: {h2['kruskal_pvalue']:.6g}")
print(f"\nResultado H2: {'VALIDADA' if h2_valid else 'PARCIAL / NO EVIDENCIA'}")


## Hipótesis 3 - Dos poblaciones en extremos de duración+título

**Hipótesis**: En extremos de duración coexisten contenidos de naturaleza distinta (musical vs no musical).


In [ ]:
# Preparación H3
h3_df = interim_df.merge(
    raw_df[["track_id", "track_name", "artist_name", "playlist_name"]],
    on="track_id",
    how="left",
    suffixes=("", "_raw"),
)

# Resolver nombres si existen duplicados por merge
if "track_name_raw" in h3_df.columns:
    h3_df["track_name"] = h3_df["track_name_raw"].fillna(h3_df.get("track_name"))
if "artist_name_raw" in h3_df.columns:
    h3_df["artist_name"] = h3_df["artist_name_raw"].fillna(h3_df.get("artist_name"))
if "playlist_name_raw" in h3_df.columns:
    h3_df["playlist_name"] = h3_df["playlist_name_raw"].fillna(h3_df.get("playlist_name"))

h3_df = tag_duration_extremes(h3_df)

extreme_pct = h3_df["extreme_flag"].mean() * 100
print(f"Registros extremos: {h3_df['extreme_flag'].sum():,} ({extreme_pct:.3f}%)")
print(h3_df[["extreme_short", "extreme_long", "extreme_flag"]].mean().mul(100).round(3))


In [ ]:
# Muestra manual asistida (reproducible)
extreme_sample = (
    h3_df.loc[h3_df["extreme_flag"], ["track_name", "artist_name", "duration_min", "playlist_name"]]
    .sample(n=min(30, h3_df["extreme_flag"].sum()), random_state=42)
    .sort_values("duration_min")
)

print("Muestra de extremos (n=30 máx):")
display(extreme_sample)


In [ ]:
# Heurística de contenido no musical por regex en título
regex_non_music = r"podcast|episodio|episode|entrevista|radio|capitulo|programa|#VueltaYMedia|meditaci[oó]n|audiolibro"
h3_df["non_music_title_flag"] = h3_df["track_name"].fillna("").str.contains(regex_non_music, case=False, regex=True)

heur_tbl = pd.DataFrame(
    {
        "grupo": ["Extremos", "No extremos"],
        "n": [int(h3_df["extreme_flag"].sum()), int((~h3_df["extreme_flag"]).sum())],
        "pct_non_music_title": [
            h3_df.loc[h3_df["extreme_flag"], "non_music_title_flag"].mean() * 100,
            h3_df.loc[~h3_df["extreme_flag"], "non_music_title_flag"].mean() * 100,
        ],
    }
)

display(heur_tbl)


In [ ]:
# Comparativa de desempeño con/sin extremos (LR)
final_features = [
    "artist_avg_popularity",
    "playlist_avg_popularity",
    "playlist_popularity_std",
    "artist_popularity_std",
    "years_since_release",
    "album_release_year",
    "playlist_track_count",
    "artist_track_count",
    "artist_avg_duration",
    "playlist_avg_duration",
]

def eval_lr_once(df, feat_cols, seed=42):
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(split.split(df, df["popular_high"]))
    tr = df.iloc[tr_idx].copy()
    te = df.iloc[te_idx].copy()

    Xtr = tr[feat_cols].copy()
    Xte = te[feat_cols].copy()
    ytr = tr["popular_high"].astype(int)
    yte = te["popular_high"].astype(int)

    for c in feat_cols:
        med = Xtr[c].median()
        Xtr[c] = Xtr[c].fillna(med)
        Xte[c] = Xte[c].fillna(med)

    model = LogisticRegression(max_iter=2000, random_state=seed)
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]

    return {
        "roc_auc": roc_auc_score(yte, p),
        "pr_auc": average_precision_score(yte, p),
        "n_rows": len(df),
    }

full_metrics = eval_lr_once(h3_df, final_features, seed=42)
no_ext_metrics = eval_lr_once(h3_df.loc[~h3_df["extreme_flag"]].copy(), final_features, seed=42)

h3_perf = pd.DataFrame([
    {"dataset": "Completo", **full_metrics},
    {"dataset": "Sin extremos", **no_ext_metrics},
])

display(h3_perf)


In [ ]:
# Decisión H3
delta_auc = h3_perf.loc[h3_perf["dataset"] == "Sin extremos", "roc_auc"].iloc[0] - h3_perf.loc[h3_perf["dataset"] == "Completo", "roc_auc"].iloc[0]
heur_ext = heur_tbl.loc[heur_tbl["grupo"] == "Extremos", "pct_non_music_title"].iloc[0]
heur_non = heur_tbl.loc[heur_tbl["grupo"] == "No extremos", "pct_non_music_title"].iloc[0]

h3_valid = bool((heur_ext > heur_non) or (delta_auc > 0.002))

print(f"Delta AUC (sin extremos - completo): {delta_auc:.6f}")
print(f"% títulos no-musicales en extremos: {heur_ext:.3f}%")
print(f"% títulos no-musicales en no extremos: {heur_non:.3f}%")
print(f"\nResultado H3: {'VALIDADA' if h3_valid else 'PARCIAL / NO EVIDENCIA'}")


## Hipótesis 4 - Valor de temporales + agregados con control anti-leakage

**Hipótesis**: El mayor valor predictivo viene de combinar temporales de álbum con agregados por entidad, pero parte de la señal se infla si se calculan globalmente (leakage).


In [ ]:
# Dataset base para H4
h4_df = interim_df.copy()

# Completar IDs desde raw solo si faltan columnas o hay nulos
need_artist = "artist_id" not in h4_df.columns
need_playlist = "playlist_id" not in h4_df.columns

if need_artist or need_playlist or h4_df[[c for c in ["artist_id", "playlist_id"] if c in h4_df.columns]].isna().any().any():
    raw_ids = raw_df[["track_id", "artist_id", "playlist_id"]].drop_duplicates("track_id")

    if need_artist or need_playlist:
        h4_df = h4_df.merge(raw_ids, on="track_id", how="left", suffixes=("", "_raw"))
    else:
        h4_df = h4_df.merge(raw_ids, on="track_id", how="left", suffixes=("", "_raw"))
        h4_df["artist_id"] = h4_df["artist_id"].fillna(h4_df.get("artist_id_raw"))
        h4_df["playlist_id"] = h4_df["playlist_id"].fillna(h4_df.get("playlist_id_raw"))

    for c in ["artist_id_raw", "playlist_id_raw"]:
        if c in h4_df.columns:
            h4_df = h4_df.drop(columns=[c])

required_h4_ids = {"artist_id", "playlist_id"}
missing_h4_ids = required_h4_ids - set(h4_df.columns)
if missing_h4_ids:
    raise ValueError(f"Faltan columnas para agregados train-only: {sorted(missing_h4_ids)}")

# Features por variante
v1_base_temporal = [
    "years_since_release",
    "album_release_year",
    "artist_track_count",
    "playlist_track_count",
    "artist_avg_duration",
    "playlist_avg_duration",
]

v2_global = [
    "years_since_release",
    "album_release_year",
    "artist_avg_popularity",
    "playlist_avg_popularity",
    "artist_popularity_std",
    "playlist_popularity_std",
    "artist_track_count",
    "playlist_track_count",
    "artist_avg_duration",
    "playlist_avg_duration",
]

v3_train_only = [
    "years_since_release",
    "album_release_year",
    "artist_avg_popularity_train",
    "playlist_avg_popularity_train",
    "artist_popularity_std_train",
    "playlist_popularity_std_train",
    "artist_track_count_train",
    "playlist_track_count_train",
    "artist_avg_duration_train",
    "playlist_avg_duration_train",
]

variants = {
    "V1_base_temporal": v1_base_temporal,
    "V2_con_agregados_globales": v2_global,
    "V3_con_agregados_train_only": v3_train_only,
}

print("Variantes H4:")
for k, v in variants.items():
    print(f"- {k}: {len(v)} features")


In [ ]:
# Evaluación CV completa (5 splits, 2 modelos)
h4_results = evaluate_model_variants_cv(h4_df, variants=variants, n_splits=5, random_state=42)

print("Resultados por split:")
display(h4_results.head(12))

h4_summary = (
    h4_results.groupby(["variant", "model"], as_index=False)
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
    )
    .sort_values(["model", "roc_auc_mean"], ascending=[True, False])
)

print("Resumen (mean/std):")
display(h4_summary)


In [ ]:
# Gap de leakage: V2 - V3
pivot_h4 = h4_results.pivot_table(index=["split_id", "model"], columns="variant", values=["roc_auc", "pr_auc"])
pivot_h4.columns = [f"{a}_{b}" for a, b in pivot_h4.columns]
pivot_h4 = pivot_h4.reset_index()

pivot_h4["gap_roc_auc_v2_minus_v3"] = pivot_h4["roc_auc_V2_con_agregados_globales"] - pivot_h4["roc_auc_V3_con_agregados_train_only"]
pivot_h4["gap_pr_auc_v2_minus_v3"] = pivot_h4["pr_auc_V2_con_agregados_globales"] - pivot_h4["pr_auc_V3_con_agregados_train_only"]

print("Gap por split/modelo:")
display(pivot_h4[["split_id", "model", "gap_roc_auc_v2_minus_v3", "gap_pr_auc_v2_minus_v3"]])

gap_summary = (
    pivot_h4.groupby("model", as_index=False)
    .agg(
        gap_roc_auc_mean=("gap_roc_auc_v2_minus_v3", "mean"),
        gap_roc_auc_std=("gap_roc_auc_v2_minus_v3", "std"),
        gap_pr_auc_mean=("gap_pr_auc_v2_minus_v3", "mean"),
        gap_pr_auc_std=("gap_pr_auc_v2_minus_v3", "std"),
    )
)

print("Resumen de gap leakage (V2 - V3):")
display(gap_summary)


In [ ]:
# Visualizaciones H4
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=h4_results, x="variant", y="roc_auc", hue="model", ax=axes[0])
axes[0].set_title("ROC-AUC por variante")
axes[0].set_xlabel("Variante")
axes[0].set_ylabel("ROC-AUC")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(alpha=0.3)

sns.barplot(data=gap_summary.melt(id_vars="model", value_vars=["gap_roc_auc_mean", "gap_pr_auc_mean"]),
            x="model", y="value", hue="variable", ax=axes[1])
axes[1].set_title("Gap promedio de leakage (V2 - V3)")
axes[1].set_xlabel("Modelo")
axes[1].set_ylabel("Gap")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Decisión H4
decision_rows = []

for model_name in sorted(h4_results["model"].unique()):
    subset = h4_results[h4_results["model"] == model_name]

    v1_auc = subset[subset["variant"] == "V1_base_temporal"]["roc_auc"].mean()
    v2_auc = subset[subset["variant"] == "V2_con_agregados_globales"]["roc_auc"].mean()
    v3_auc = subset[subset["variant"] == "V3_con_agregados_train_only"]["roc_auc"].mean()

    v1_pr = subset[subset["variant"] == "V1_base_temporal"]["pr_auc"].mean()
    v2_pr = subset[subset["variant"] == "V2_con_agregados_globales"]["pr_auc"].mean()
    v3_pr = subset[subset["variant"] == "V3_con_agregados_train_only"]["pr_auc"].mean()

    supports_value = (v3_auc > v1_auc) and (v3_pr > v1_pr)
    supports_leakage = (v2_auc > v3_auc) and (v2_pr > v3_pr)
    h4_model_valid = supports_value and supports_leakage

    decision_rows.append({
        "model": model_name,
        "v1_auc": v1_auc,
        "v3_auc": v3_auc,
        "v2_auc": v2_auc,
        "v1_pr": v1_pr,
        "v3_pr": v3_pr,
        "v2_pr": v2_pr,
        "value_signal(v3>v1)": supports_value,
        "leakage_signal(v2>v3)": supports_leakage,
        "decision_model": "VALIDADA" if h4_model_valid else "PARCIAL/NO EVIDENCIA",
    })

h4_decision_df = pd.DataFrame(decision_rows)
display(h4_decision_df)

h4_valid = bool((h4_decision_df["decision_model"] == "VALIDADA").all())
print(f"Resultado H4: {'VALIDADA' if h4_valid else 'PARCIAL / NO EVIDENCIA'}")


## Conclusiones por hipótesis


In [ ]:
# Consolidado final
status_h1 = "VALIDADA" if h1_valid else "PARCIAL / NO EVIDENCIA"
status_h2 = "VALIDADA" if h2_valid else "PARCIAL / NO EVIDENCIA"
status_h3 = "VALIDADA" if h3_valid else "PARCIAL / NO EVIDENCIA"
status_h4 = "VALIDADA" if h4_valid else "PARCIAL / NO EVIDENCIA"

final_summary = pd.DataFrame([
    {
        "hipotesis": "H1 - Nulos estructurales por source",
        "evidencia_principal": "Asociación source vs presencia de search_*/playlist_*/added_at (chi2 + Cramér's V)",
        "metrica_clave": f"min p-value={h1_stats_df['pvalue'].min():.3e}; min Cramér's V={h1_stats_df['cramers_v'].min():.3f}",
        "decision": status_h1,
    },
    {
        "hipotesis": "H2 - Señal no lineal duración-popularidad",
        "evidencia_principal": "MI + Spearman vs Pearson + Kruskal por bins",
        "metrica_clave": f"MI={h2['mutual_information']:.6f}; Kruskal p={h2['kruskal_pvalue']:.3e}",
        "decision": status_h2,
    },
    {
        "hipotesis": "H3 - Dos poblaciones en extremos",
        "evidencia_principal": "Muestra manual + regex no-musical + desempeño con/sin extremos",
        "metrica_clave": f"delta_auc={delta_auc:.6f}; non_music_extremos={heur_ext:.2f}%",
        "decision": status_h3,
    },
    {
        "hipotesis": "H4 - Agregados con control anti-leakage",
        "evidencia_principal": "CV 5 splits con V1 vs V2 vs V3 en 2 modelos",
        "metrica_clave": f"gap_roc_auc_mean(LR)={gap_summary.loc[gap_summary['model']=='LogisticRegression','gap_roc_auc_mean'].iloc[0]:.6f}",
        "decision": status_h4,
    },
])

display(final_summary)

print("\nPróximos pasos sugeridos para 03_modeling.ipynb:")
print("1) Calcular agregados por entidad solo con train fold (anti-leakage).")
print("2) Evaluar sensibilidad del modelo con y sin extremos de duración.")
print("3) Mantener monitoreo por source para evitar sesgos por contexto de captura.")
